--- Day 22: Monkey Market ---
As you're all teleported deep into the jungle, a monkey steals The Historians' device! You'll need to get it back while The Historians are looking for the Chief.

The monkey that stole the device seems willing to trade it, but only in exchange for an absurd number of bananas. Your only option is to buy bananas on the Monkey Exchange Market.

You aren't sure how the Monkey Exchange Market works, but one of The Historians senses trouble and comes over to help. Apparently, they've been studying these monkeys for a while and have deciphered their secrets.

Today, the Market is full of monkeys buying good hiding spots. Fortunately, because of the time you recently spent in this jungle, you know lots of good hiding spots you can sell! If you sell enough hiding spots, you should be able to get enough bananas to buy the device back.

On the Market, the buyers seem to use random prices, but their prices are actually only pseudorandom! If you know the secret of how they pick their prices, you can wait for the perfect time to sell.

The part about secrets is literal, the Historian explains. Each buyer produces a pseudorandom sequence of secret numbers where each secret is derived from the previous.

In particular, each buyer's secret number evolves into the next secret number in the sequence via the following process:

Calculate the result of multiplying the secret number by 64. Then, mix this result into the secret number. Finally, prune the secret number.
Calculate the result of dividing the secret number by 32. Round the result down to the nearest integer. Then, mix this result into the secret number. Finally, prune the secret number.
Calculate the result of multiplying the secret number by 2048. Then, mix this result into the secret number. Finally, prune the secret number.
Each step of the above process involves mixing and pruning:

To mix a value into the secret number, calculate the bitwise XOR of the given value and the secret number. Then, the secret number becomes the result of that operation. (If the secret number is 42 and you were to mix 15 into the secret number, the secret number would become 37.)
To prune the secret number, calculate the value of the secret number modulo 16777216. Then, the secret number becomes the result of that operation. (If the secret number is 100000000 and you were to prune the secret number, the secret number would become 16113920.)
After this process completes, the buyer is left with the next secret number in the sequence. The buyer can repeat this process as many times as necessary to produce more secret numbers.

So, if a buyer had a secret number of 123, that buyer's next ten secret numbers would be:

15887950
16495136
527345
704524
1553684
12683156
11100544
12249484
7753432
5908254
Each buyer uses their own secret number when choosing their price, so it's important to be able to predict the sequence of secret numbers for each buyer. Fortunately, the Historian's research has uncovered the initial secret number of each buyer (your puzzle input). For example:

1
10
100
2024
This list describes the initial secret number of four different secret-hiding-spot-buyers on the Monkey Exchange Market. If you can simulate secret numbers from each buyer, you'll be able to predict all of their future prices.

In a single day, buyers each have time to generate 2000 new secret numbers. In this example, for each buyer, their initial secret number and the 2000th new secret number they would generate are:

1: 8685429
10: 4700978
100: 15273692
2024: 8667524
Adding up the 2000th new secret number for each buyer produces 37327623.

For each buyer, simulate the creation of 2000 new secret numbers. What is the sum of the 2000th secret number generated by each buyer?

In [1]:
from math import log2
log2(64), log2(32), log2(2048), log2(16777216)

(6.0, 5.0, 11.0, 24.0)

In [2]:
with open("input22.txt") as file:
    data = [int(num) for num in file]
max(data) < 2 ** 24

True

In [3]:
"""
What are the operations?

First modification:
We add 6 0-bits (* 64)
We XOR the numbers (mix)
We retain the last 24 bits (prune)

Second modification:
We remove the last 5 bits (// 32)
We XOR the numbers (mix)
We retain the last 24 bits (prune)

Third modification:
Add 11 0-bits (* 2048)
We XOR the numbers (mix)
We retain the last 24 bits (prune)

So basically we work with 24 bit numbers (all puzzle input numbers are within 24 bits as well)

We bit-shift to the left or the right and calculate a XOR. This means that for multiplication with 2 ** x, the first and last x bits are retained, while the middle bits (24 - x) are mixed with an offset of x. However, after the prune we basically end with mixed bits + originals at the end

Multiplication (8-bit):
xxx12345678
xxx45678000

Floor division is the same thing from the other end, where the first X bits are retained, while the remaining bits are mixed with an offset of x

Floor division (8-bit):
    1234
12345678

What happens as we combine operations?
After the first modification, we have a 24bit number where the first 18 bits are mixed (as 1^7, 2^8 etc) and the final six bits are original again

The second modification retains the first 5 bits while mixing all others

The last modification retains the last 11 bits while mixing the others. 
"""

'\nWhat are the operations?\n\nFirst modification:\nWe add 6 0-bits (* 64)\nWe XOR the numbers (mix)\nWe retain the last 24 bits (prune)\n\nSecond modification:\nWe remove the last 5 bits (// 32)\nWe XOR the numbers (mix)\nWe retain the last 24 bits (prune)\n\nThird modification:\nAdd 11 0-bits (* 2048)\nWe XOR the numbers (mix)\nWe retain the last 24 bits (prune)\n\nSo basically we work with 24 bit numbers (all puzzle input numbers are within 24 bits as well)\n\nWe bit-shift to the left or the right and calculate a XOR. This means that for multiplication with 2 ** x, the first and last x bits are retained, while the middle bits (24 - x) are mixed with an offset of x. However, after the prune we basically end with mixed bits + originals at the end\n\nMultiplication (8-bit):\nxxx12345678\nxxx45678000\n\nFloor division is the same thing from the other end, where the first X bits are retained, while the remaining bits are mixed with an offset of x\n\nFloor division (8-bit):\n    1234\n123

In [4]:
# Let's create a little helper that allows us to see how each bit in the final secret number is computed:

bits = {x : [x] for x in range(24)}

def compose_full(ledger, pow2):
    # hard-code multiply/divide by power of two, mix and prune operations. Pow2 positive (for mult) or negative (for div)
    if pow2 > 0:
        for i in range(24 - pow2):
            ledger[i] += ledger[i + pow2]
    else:
        shift = -pow2
        for i in range(23, shift - 1, -1):
            ledger[i] += ledger[i - shift]
compose_full(bits, 6)
compose_full(bits, -5)
compose_full(bits, 11)
bits

{0: [0, 6, 11, 17, 6, 12],
 1: [1, 7, 12, 18, 7, 13],
 2: [2, 8, 13, 19, 8, 14],
 3: [3, 9, 14, 20, 9, 15],
 4: [4, 10, 15, 21, 10, 16],
 5: [5, 11, 0, 6, 16, 22, 11, 17],
 6: [6, 12, 1, 7, 17, 23, 12, 18],
 7: [7, 13, 2, 8, 18, 13, 19],
 8: [8, 14, 3, 9, 19, 14, 20],
 9: [9, 15, 4, 10, 20, 15, 21],
 10: [10, 16, 5, 11, 21, 16, 22],
 11: [11, 17, 6, 12, 22, 17, 23],
 12: [12, 18, 7, 13, 23, 18],
 13: [13, 19, 8, 14],
 14: [14, 20, 9, 15],
 15: [15, 21, 10, 16],
 16: [16, 22, 11, 17],
 17: [17, 23, 12, 18],
 18: [18, 13, 19],
 19: [19, 14, 20],
 20: [20, 15, 21],
 21: [21, 16, 22],
 22: [22, 17, 23],
 23: [23, 18]}

In [5]:
# Now for calculating the puzzle results, we can do this: We can automatically remove duplicate bits and then chain all 2000 secret number calculations to arrive at the final number
from functools import cache

def compose_lean(ledger, pow2):
    if pow2 > 0:
        for i in range(24 - pow2):
            ledger[i] ^= ledger[i + pow2]
    else:
        shift = -pow2
        for i in range(23, shift - 1, -1):
            ledger[i] ^= ledger[i - shift]

@cache
def compose_n(n):
    bits = {x : set([x]) for x in range(24)}
    for _ in range(n):
        compose_lean(bits, 6)
        compose_lean(bits, -5)
        compose_lean(bits, 11)
    return bits

secret_2000 = compose_n(2000)
secret_2000

{0: {2, 4, 5, 7, 10, 13, 14, 15, 16, 17, 23},
 1: {0, 1, 3, 4, 7, 9, 13, 15, 17, 18, 19, 21, 22},
 2: {0, 2, 5, 6, 10, 12, 16, 17, 19},
 3: {4, 5, 7, 8, 10, 11, 12, 13, 16, 17, 20, 21, 22},
 4: {0, 2, 5, 7, 9, 10, 12, 13, 16, 18, 19, 20, 22},
 5: {1, 2, 6, 9, 12, 13, 14, 15, 18, 19, 20, 21, 23},
 6: {0, 5, 9, 10, 11, 15, 17, 21, 22},
 7: {1, 2, 3, 7, 9, 13, 14, 19, 21, 22},
 8: {0, 3, 5, 7, 8, 9, 12, 13, 14, 15, 16, 18, 20, 21, 23},
 9: {5, 6, 7, 8, 15, 19},
 10: {1, 2, 3, 6, 7, 8, 9, 10, 12, 13, 16, 17, 18, 22},
 11: {0, 4, 5, 8, 10, 12, 13, 15, 17, 18, 22},
 12: {0, 2, 6, 8, 10, 11, 13, 14, 16, 19, 20, 21},
 13: {0, 3, 5, 6, 10, 11, 13, 14, 15, 17, 23},
 14: {1, 3, 5, 6, 7, 10, 11, 14, 15, 19, 21, 22, 23},
 15: {1, 3, 5, 9, 11, 13, 15, 19, 20, 23},
 16: {1, 2, 4, 10, 12, 18, 19},
 17: {0, 4, 9, 10, 11, 13, 14, 15, 17, 23},
 18: {3, 4, 5, 8, 11, 12, 14, 15, 17, 18, 22, 23},
 19: {5, 6, 7, 11, 13, 16, 17, 20, 23},
 20: {0, 2, 3, 5, 6, 10, 16, 17, 19},
 21: {0, 3, 6, 7, 9, 14, 16, 20, 2

In [6]:
def secret_num(seed, n):
    mapping = compose_n(n)
    seed_bits = [int(i) for i in f'{seed:024b}']
    bit_list = []
    for i in range(24):
        mapped_bit = sum([seed_bits[j] for j in mapping[i]]) % 2
        bit_list.append(mapped_bit)
    num = int(''.join(str(b) for b in bit_list), 2)
    return num
secret_num(2024, 2000)

8667524

In [7]:
sum_num = 0
for num in data:
    sum_num += secret_num(num, 2000)
sum_num

16039090236

Your puzzle answer was 16039090236.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
Of course, the secret numbers aren't the prices each buyer is offering! That would be ridiculous. Instead, the prices the buyer offers are just the ones digit of each of their secret numbers.

So, if a buyer starts with a secret number of 123, that buyer's first ten prices would be:

3 (from 123)
0 (from 15887950)
6 (from 16495136)
5 (etc.)
4
4
6
4
4
2
This price is the number of bananas that buyer is offering in exchange for your information about a new hiding spot. However, you still don't speak monkey, so you can't negotiate with the buyers directly. The Historian speaks a little, but not enough to negotiate; instead, he can ask another monkey to negotiate on your behalf.

Unfortunately, the monkey only knows how to decide when to sell by looking at the changes in price. Specifically, the monkey will only look for a specific sequence of four consecutive changes in price, then immediately sell when it sees that sequence.

So, if a buyer starts with a secret number of 123, that buyer's first ten secret numbers, prices, and the associated changes would be:

     123: 3 
15887950: 0 (-3)
16495136: 6 (6)
  527345: 5 (-1)
  704524: 4 (-1)
 1553684: 4 (0)
12683156: 6 (2)
11100544: 4 (-2)
12249484: 4 (0)
 7753432: 2 (-2)
Note that the first price has no associated change because there was no previous price to compare it with.

In this short example, within just these first few prices, the highest price will be 6, so it would be nice to give the monkey instructions that would make it sell at that time. The first 6 occurs after only two changes, so there's no way to instruct the monkey to sell then, but the second 6 occurs after the changes -1,-1,0,2. So, if you gave the monkey that sequence of changes, it would wait until the first time it sees that sequence and then immediately sell your hiding spot information at the current price, winning you 6 bananas.

Each buyer only wants to buy one hiding spot, so after the hiding spot is sold, the monkey will move on to the next buyer. If the monkey never hears that sequence of price changes from a buyer, the monkey will never sell, and will instead just move on to the next buyer.

Worse, you can only give the monkey a single sequence of four price changes to look for. You can't change the sequence between buyers.

You're going to need as many bananas as possible, so you'll need to determine which sequence of four price changes will cause the monkey to get you the most bananas overall. Each buyer is going to generate 2000 secret numbers after their initial secret number, so, for each buyer, you'll have 2000 price changes in which your sequence can occur.

Suppose the initial secret number of each buyer is:

1
2
3
2024
There are many sequences of four price changes you could tell the monkey, but for these four buyers, the sequence that will get you the most bananas is -2,1,-1,3. Using that sequence, the monkey will make the following sales:

For the buyer with an initial secret number of 1, changes -2,1,-1,3 first occur when the price is 7.
For the buyer with initial secret 2, changes -2,1,-1,3 first occur when the price is 7.
For the buyer with initial secret 3, the change sequence -2,1,-1,3 does not occur in the first 2000 changes.
For the buyer starting with 2024, changes -2,1,-1,3 first occur when the price is 9.
So, by asking the monkey to sell the first time each buyer's prices go down 2, then up 1, then down 1, then up 3, you would get 23 (7 + 7 + 9) bananas!

Figure out the best sequence to tell the monkey so that by looking for that same sequence of changes in every buyer's future prices, you get the most bananas in total. What is the most bananas you can get?

In [8]:
"""
I was hoping my heavily optimized way of approaching calculating the n-th secret number would help me out in part 2. Alas, part 2 goes in a different direction. 

In fact, it might not be a bitwise problem at all (it might in subtle ways). On the surface, we have rolling sequences of single digit integer numbers and want to find a sequence of 4 (looking at the relative changes) that reliably leads to large final digits. Also note that a final digit of 0 to 7 corresponds to a 4 bit number, but an 8 or 9 corresponds to a 5 bit number!

In addition, if we look at the binary level, our XOR-maps show how numbers change between iterations, however it is a XOR pattern - it depends entirely on the initial seed if a particular sequence of XOR changes lead to a high or low final output. So my assumption is that the bit-wise level is not that significant here. 

"""

'\nI was hoping my heavily optimized way of approaching calculating the n-th secret number would help me out in part 2. Alas, part 2 goes in a different direction. \n\nIn fact, it might not be a bitwise problem at all (it might in subtle ways). On the surface, we have rolling sequences of single digit integer numbers and want to find a sequence of 4 (looking at the relative changes) that reliably leads to large final digits. Also note that a final digit of 0 to 7 corresponds to a 4 bit number, but an 8 or 9 corresponds to a 5 bit number!\n\nIn addition, if we look at the binary level, our XOR-maps show how numbers change between iterations, however it is a XOR pattern - it depends entirely on the initial seed if a particular sequence of XOR changes lead to a high or low final output. So my assumption is that the bit-wise level is not that significant here. \n\n'

In [9]:
def seq_prize(seed):
    seq = []
    sec = seed
    prize = sec % 10
    for _ in range(4):
        new_sec = secret_num(sec, 1)
        new_prize = new_sec % 10
        seq.append(new_prize - prize)
        sec, prize = new_sec, new_prize
    return (seq, prize)
seq_prize(16495136)

([-1, -1, 0, 2], 6)

In [12]:
"""
Let's generate random initial numbers and see if we can spot some trends
"""
from random import randint
def rand_seed():
    return secret_num(randint(0, 2**24 -1), 2) # wash it a bit for typicality

randoms = []
for _ in range(100000):
    randoms.append(seq_prize(rand_seed()))
len(randoms), randoms[:5]

(100000,
 [([1, 3, -5, 2], 5),
  ([-1, 2, -7, 1], 1),
  ([-5, 0, 2, 3], 9),
  ([7, -2, 1, -5], 3),
  ([1, -2, 4, 2], 9)])

In [13]:
import polars as pl
sample_data = {"prize" : [i[1] for i in randoms],
               "change_1" : [i[0][0] for i in randoms],
               "change_2" : [i[0][1] for i in randoms],
               "change_3" : [i[0][2] for i in randoms],
               "change_4" : [i[0][3] for i in randoms],}

sample_df = pl.DataFrame(sample_data)
sample_df.head()

prize,change_1,change_2,change_3,change_4
i64,i64,i64,i64,i64
5,1,3,-5,2
1,-1,2,-7,1
9,-5,0,2,3
3,7,-2,1,-5
9,1,-2,4,2


In [14]:
sample_df.group_by('prize').len().sort('prize') # all prizes appear roughly equally often

prize,len
i64,u32
0,10145
1,9997
2,9932
3,9924
4,9982
5,10083
6,10016
7,9947
8,9943


In [15]:
pl.Config.set_tbl_rows(100)
sample_df.group_by('change_4').len().sort('change_4') # larger changes are rarer

change_4,len
i64,u32
-9,1062
-8,2029
-7,3001
-6,4088
-5,5010
-4,6031
-3,7028
-2,7838
-1,9052


In [16]:
sample_df.filter(pl.col('prize') == 9).group_by('change_4').len().sort('change_4') # However, if we aim for a final prize of 9, the distribution is roughly equal

change_4,len
i64,u32
0,1021
1,961
2,1049
3,993
4,977
5,1064
6,1001
7,956
8,1038


In [17]:
# sanity check: what if the final prize is 5?
sample_df.filter(pl.col('prize') == 5).group_by('change_4').len().sort('change_4')

change_4,len
i64,u32
-4,1019
-3,1043
-2,981
-1,1045
0,992
1,1019
2,1013
3,960
4,988


In [18]:
# Ok. Now what are the most common distributions ending in a 9?
df_nines = sample_df.filter(pl.col('prize') == 9)
df_nines.group_by(df_nines).len().sort('len', descending=True).head()

prize,change_1,change_2,change_3,change_4,len
i64,i64,i64,i64,i64,u32
9,-8,3,0,5,7
9,4,-7,5,3,6
9,4,-2,-2,5,6
9,4,-5,8,0,6
9,0,0,3,1,6


In [19]:
df_nines = df_nines.with_columns(start_prize = pl.col('prize') - pl.col('change_4') - pl.col('change_3') - pl.col('change_2') - pl.col('change_1'))
df_nines.group_by('start_prize').len().sort('start_prize')

start_prize,len
i64,u32
0,970
1,970
2,984
3,997
4,995
5,1064
6,996
7,992
8,1007


In [20]:
df_nines.filter(pl.col('start_prize') == 0).group_by(["change_1", "change_2", "change_3", "change_4"]).len().sort('len', descending=True).head(20)

change_1,change_2,change_3,change_4,len
i64,i64,i64,i64,u32
0,1,3,5,6
5,-3,7,0,5
7,1,-1,2,5
0,7,-6,8,5
7,2,-4,4,4
0,2,0,7,4
6,-1,0,4,4
4,0,-1,6,4
1,6,-3,5,4


In [21]:
"""
Well I don't think the statistical exploration has shown anything conclusive besides that it is a reasonable assumption that our final prize we aim for should be a 9 (the highest possible payoff) since all prizes are roughly equally common and that we should consider having a 0 prize somewhere in the sequence to guarantee that 9 payoff

Other than that, individual prizes seem to be mostly uncorrelated
"""

p_df = sample_df.with_columns(p3 = pl.col('prize') - pl.col('change_4'),
                                   p2 = pl.col('prize') - pl.col('change_4') - pl.col('change_3'),
                                   p1 = pl.col('prize') - pl.col('change_4') - pl.col('change_3') - pl.col('change_2'),
                                   p0 = pl.col('prize') - pl.col('change_4') - pl.col('change_3') - pl.col('change_2') - pl.col('change_1'))
p_df = p_df.select(['p0', 'p1', 'p2', 'p3', 'prize'])
p_df.corr()

p0,p1,p2,p3,prize
f64,f64,f64,f64,f64
1.0,0.003883,0.000603,-0.00203,0.000717
0.003883,1.0,-0.000877,-0.004526,-0.003255
0.000603,-0.000877,1.0,0.001355,-0.002084
-0.00203,-0.004526,0.001355,1.0,-0.002562
0.000717,-0.003255,-0.002084,-0.002562,1.0


In [105]:
# Ok let's go back to optimize a more bruteforce approach.
from copy import deepcopy
from collections import defaultdict

@cache
def compose_n(n):
    if n > 1:
        bits = deepcopy(compose_n(n-1))
    else:
        bits = {x : set([x]) for x in range(24)}
        if n == 0:
            return bits

    compose_lean(bits, 6)
    compose_lean(bits, -5)
    compose_lean(bits, 11)

    return bits

@cache
def bitmask_n(n):
    bit_masks = []
    for ix_out in range(24):
        num_str = ''
        for mask_bit in range(24):
            if mask_bit in compose_n(n)[ix_out]:
                num_str += '1'
            else:
                num_str += '0'
        bit_masks.append(int(num_str, 2))
    return bit_masks

def secret_num(seed, n):
    result = 0
    for i in range(24):
        bit = (seed & bitmask_n(n)[i]).bit_count() & 1
        result |= bit << 23 - i
    return result

def prize(seed, n):
    return secret_num(seed, n) % 10

def one_merchant(seed):
    prizes = [prize(seed, n) for n in range(2001)]
    changes = [p1 - p0 for p0, p1 in zip(prizes, prizes[1:])]
    ch_4 = [(changes[i-3], changes[i-2], changes[i-1], changes[i]) for i in range(3, len(changes))]
    seq_dict = {}
    for c,p in zip(ch_4, prizes[4:]):
        seq_dict.setdefault(c, p)

    return seq_dict

In [106]:
test_data = [1, 2, 3, 2024]

In [107]:
all_merchants = []
for merchant_seed in test_data:
    all_merchants.append(one_merchant(merchant_seed))

In [108]:
# iterate through sequences:
all_seq = set(seq for merchant in all_merchants for seq in merchant)
seq_ban = {}
for seq in all_seq:
    sum_ban = 0
    for merchant in all_merchants:
        sum_ban += merchant.setdefault(seq, 0)
    seq_ban[seq] = sum_ban

In [109]:
max(seq_ban, key=seq_ban.get)

(-2, 1, -1, 3)

In [110]:
seq_ban[(-2, 1, -1, 3)]

23

In [111]:
with open("input22.txt") as file:
    data = [int(num) for num in file]
all_merchants = []
for merchant_seed in data:
    all_merchants.append(one_merchant(merchant_seed))

In [ ]:
# iterate through sequences:
all_seq = set(seq for merchant in all_merchants for seq in merchant)
seq_ban = {}
for seq in all_seq:
    sum_ban = 0
    for merchant in all_merchants:
        sum_ban += merchant.get(seq, 0)
    seq_ban[seq] = sum_ban

In [113]:
max(seq_ban, key=seq_ban.get)

(-2, 3, -3, 3)

In [114]:
seq_ban[(-2, 3, -3, 3)]

1808

In [115]:
dict(sorted(seq_ban.items(), key=lambda x: x[1], reverse=True))

{(-2, 3, -3, 3): 1808,
 (0, 1, 0, 0): 1785,
 (-2, 2, -2, 2): 1726,
 (1, 1, 0, 0): 1723,
 (2, 0, -1, 2): 1719,
 (-2, 0, 1, 1): 1717,
 (0, 0, -1, 3): 1715,
 (0, -2, 1, 1): 1704,
 (2, -1, -1, 2): 1700,
 (-1, -1, 1, 1): 1700,
 (-1, -1, 1, 2): 1692,
 (-1, -1, 2, 0): 1672,
 (-1, 1, -1, 2): 1670,
 (1, 0, 0, 0): 1669,
 (-1, 0, 1, 0): 1665,
 (1, -1, 2, 0): 1661,
 (-1, -1, -1, 3): 1659,
 (0, -3, 1, 2): 1658,
 (3, -4, 1, 4): 1654,
 (1, -2, 1, 1): 1651,
 (-1, 0, 1, 1): 1649,
 (0, 0, 1, 2): 1646,
 (1, 0, -1, 1): 1644,
 (1, -1, -1, 2): 1644,
 (0, -2, 0, 3): 1639,
 (-1, 1, 2, 0): 1637,
 (0, 2, 0, 1): 1636,
 (0, -3, 3, 1): 1636,
 (-1, -2, 2, 2): 1632,
 (-2, 3, -1, 1): 1631,
 (-1, 2, 0, 0): 1630,
 (-2, 2, 0, 1): 1628,
 (0, -1, 1, 1): 1626,
 (0, 4, -3, 3): 1626,
 (1, -1, 1, 3): 1624,
 (-1, 2, -2, 2): 1624,
 (-1, 1, -1, 1): 1623,
 (-1, 0, 0, 2): 1620,
 (0, 1, 1, 0): 1620,
 (-1, 2, -2, 3): 1619,
 (0, 0, 0, 1): 1619,
 (1, 0, 0, 1): 1618,
 (0, 0, -1, 1): 1616,
 (0, 1, -1, 1): 1614,
 (1, 1, -4, 4): 1613,
 (-

That's the right answer! You are one gold star closer to finding the Chief Historian.

You have completed Day 22! You can [Share] this victory or [Return to Your Advent Calendar].
I love my baby bubu and my baby babu
